In [ ]:
pip install -q segmentation-models-pytorch

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
!nvidia-smi

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

In [ ]:
import json

ann_file = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco/train/_annotations.coco.json"

with open(ann_file, "r") as f:
    coco_data = json.load(f)

print(coco_data["categories"])

In [ ]:
from pycocotools.coco import COCO

coco = COCO("/kaggle/input/datasets/zahra313/dataskripsizahracscoco/train/_annotations.coco.json")

used_categories = set()

for ann in coco.anns.values():
    used_categories.add(ann["category_id"])

print("Kategori yang digunakan:", used_categories)

## EDA

In [ ]:
from collections import Counter
from pycocotools.coco import COCO

coco = COCO("/kaggle/input/datasets/zahra313/dataskripsizahracscoco/train/_annotations.coco.json")

counter = Counter()

for ann in coco.anns.values():
    counter[ann["category_id"]] += 1

print(counter)

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import imagesize

# Path dataset COCO Segmentation
data_dir = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco"

subsets = ['train', 'valid', 'test']

img_shapes = []

for subset in subsets:

    image_folder = os.path.join(data_dir, subset)

    if os.path.exists(image_folder):
        print(f"Memproses {subset}...")

        for file_name in os.listdir(image_folder):

            # Abaikan file anotasi COCO
            if file_name == "_annotations.coco.json":
                continue

            if file_name.lower().endswith(
                ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
            ):

                img_path = os.path.join(image_folder, file_name)

                try:
                    width, height = imagesize.get(img_path)

                    if width > 0 and height > 0:
                        img_shapes.append((width, height))

                except Exception as e:
                    print(f"Gagal membaca {file_name}: {e}")



if len(img_shapes) > 0:

    widths = [shape[0] for shape in img_shapes]
    heights = [shape[1] for shape in img_shapes]

    print(f"\nTotal gambar berhasil dianalisis : {len(img_shapes)}")

    print(f"Lebar minimum   : {min(widths)} px")
    print(f"Lebar maksimum  : {max(widths)} px")
    print(f"Lebar rata-rata : {sum(widths)/len(widths):.2f} px")

    print(f"Tinggi minimum   : {min(heights)} px")
    print(f"Tinggi maksimum  : {max(heights)} px")
    print(f"Tinggi rata-rata : {sum(heights)/len(heights):.2f} px")

    plt.figure(figsize=(14,5))

    # Distribusi Width
    plt.subplot(1,2,1)
    sns.histplot(widths, bins=20)
    plt.title("Distribusi Lebar Gambar")
    plt.xlabel("Width (px)")
    plt.ylabel("Jumlah")

    # Distribusi Height
    plt.subplot(1,2,2)
    sns.histplot(heights, bins=20)
    plt.title("Distribusi Tinggi Gambar")
    plt.xlabel("Height (px)")
    plt.ylabel("Jumlah")

    plt.tight_layout()
    plt.show()

else:
    print("Tidak ada gambar yang ditemukan.")

In [ ]:
import json

ann_file = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco/train/_annotations.coco.json"

with open(ann_file, "r") as f:
    coco_data = json.load(f)

print(coco_data.keys())

In [ ]:
from pycocotools.coco import COCO
from collections import defaultdict

ann_file = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco/train/_annotations.coco.json"

coco = COCO(ann_file)


class_images = defaultdict(set)

for ann in coco.anns.values():

    class_id = ann["category_id"]

    if class_id == 0:
        continue

    class_images[class_id].add(ann["image_id"])

# Tampilkan hasil
print("Distribusi Gambar per Kelas")
print("-" * 35)

for cat in coco.loadCats(coco.getCatIds()):

    if cat["id"] == 0:
        continue

    count = len(class_images[cat["id"]])

    print(f"{cat['name']:10s}: {count} gambar")

In [ ]:
import matplotlib.pyplot as plt

labels = []
values = []

for cat in coco.loadCats(coco.getCatIds()):

    if cat["id"] == 0:
        continue

    labels.append(cat["name"])
    values.append(len(class_images[cat["id"]]))

plt.figure(figsize=(6,4))

bars = plt.bar(labels, values)

plt.title("Distribusi Gambar per Kelas")
plt.xlabel("Kelas")
plt.ylabel("Jumlah Gambar")

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        int(bar.get_height()),
        ha='center',
        va='bottom'
    )

plt.show()

In [ ]:
from pycocotools.coco import COCO
from collections import defaultdict

ann_file = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco/valid/_annotations.coco.json"

coco = COCO(ann_file)


class_images = defaultdict(set)

for ann in coco.anns.values():

    class_id = ann["category_id"]

    if class_id == 0:
        continue

    class_images[class_id].add(ann["image_id"])


print("Distribusi Gambar per Kelas")
print("-" * 35)

for cat in coco.loadCats(coco.getCatIds()):

    if cat["id"] == 0:
        continue

    count = len(class_images[cat["id"]])

    print(f"{cat['name']:10s}: {count} gambar")

In [ ]:
import matplotlib.pyplot as plt

labels = []
values = []

for cat in coco.loadCats(coco.getCatIds()):

    if cat["id"] == 0:
        continue

    labels.append(cat["name"])
    values.append(len(class_images[cat["id"]]))

plt.figure(figsize=(6,4))

bars = plt.bar(labels, values)

plt.title("Distribusi Gambar per Kelas")
plt.xlabel("Kelas")
plt.ylabel("Jumlah Gambar")

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        int(bar.get_height()),
        ha='center',
        va='bottom'
    )

plt.show()

In [ ]:
from pycocotools.coco import COCO
from collections import defaultdict

ann_file = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco/test/_annotations.coco.json"

coco = COCO(ann_file)


class_images = defaultdict(set)

for ann in coco.anns.values():

    class_id = ann["category_id"]

    if class_id == 0:
        continue

    class_images[class_id].add(ann["image_id"])


print("Distribusi Gambar per Kelas")
print("-" * 35)

for cat in coco.loadCats(coco.getCatIds()):

    if cat["id"] == 0:
        continue

    count = len(class_images[cat["id"]])

    print(f"{cat['name']:10s}: {count} gambar")

In [ ]:
import matplotlib.pyplot as plt

labels = []
values = []

for cat in coco.loadCats(coco.getCatIds()):

    if cat["id"] == 0:
        continue

    labels.append(cat["name"])
    values.append(len(class_images[cat["id"]]))

plt.figure(figsize=(6,4))

bars = plt.bar(labels, values)

plt.title("Distribusi Gambar per Kelas")
plt.xlabel("Kelas")
plt.ylabel("Jumlah Gambar")

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        int(bar.get_height()),
        ha='center',
        va='bottom'
    )

plt.show()

In [ ]:
import json

ann_file = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco/train/_annotations.coco.json"

with open(ann_file, "r") as f:
    coco_data = json.load(f)

print("Jumlah gambar      :", len(coco_data["images"]))
print("Jumlah anotasi    :", len(coco_data["annotations"]))
print("Jumlah kategori   :", len(coco_data["categories"]))

print("\nDaftar Kategori:")
for cat in coco_data["categories"]:
    print(f"ID: {cat['id']} | Nama: {cat['name']}")

In [ ]:
import os

base_path = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco"

splits = ["train", "valid", "test"]

print("Distribusi Gambar per Folder:\n")

for split in splits:
    folder = os.path.join(base_path, split)
    images = os.listdir(folder)

    print(f"{split.upper():5} : {len(images)} gambar")

In [ ]:
import os

base = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco"

train = set(os.listdir(base + "/train"))
val   = set(os.listdir(base + "/valid"))
test  = set(os.listdir(base + "/test"))

print("Train ∩ Val :", len(train & val))
print("Train ∩ Test:", len(train & test))
print("Val ∩ Test  :", len(val & test))

In [ ]:
import os

base = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco"

train = set(os.listdir(base + "/train"))
val   = set(os.listdir(base + "/valid"))
test  = set(os.listdir(base + "/test"))

dup_train_val = train & val
dup_train_test = train & test
dup_val_test = val & test

print("Duplicate Train-Val:", dup_train_val)
print("Duplicate Train-Test:", dup_train_test)
print("Duplicate Val-Test:", dup_val_test)

In [ ]:
from pycocotools.coco import COCO
from PIL import Image
import matplotlib.pyplot as plt
import os



train_dir = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco/train"

ann_file = os.path.join(
    train_dir,
    "_annotations.coco.json"
)



coco = COCO(ann_file)


categories = coco.loadCats(coco.getCatIds())

class_names = {
    cat["id"]: cat["name"]
    for cat in categories
    if cat["id"] != 0
}

print("Kelas ditemukan:")
print(class_names)


example_images = {}

for ann in coco.anns.values():

    class_id = ann["category_id"]

    if class_id not in example_images:

        image_id = ann["image_id"]

        img_info = coco.loadImgs(image_id)[0]

        example_images[class_id] = img_info["file_name"]



num_classes = len(example_images)

plt.figure(figsize=(6 * num_classes, 6))

for idx, class_id in enumerate(sorted(example_images.keys())):

    image_file = example_images[class_id]

    img_path = os.path.join(
        train_dir,
        image_file
    )

    img = Image.open(img_path)

    plt.subplot(1, num_classes, idx + 1)

    plt.imshow(img)

    plt.title(
        f"{class_names[class_id]}"
    )
    plt.axis("off")

plt.suptitle(
    "Contoh Gambar Setiap Kelas",
    fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
#pip install pycocotools -q

In [ ]:
from pycocotools.coco import COCO
import numpy as np
import cv2
import os
import shutil

def reset_folder(path):
    """
    Hapus folder lama agar tidak terjadi duplicate mask
    """
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)


def coco_to_mask(coco_json_path, output_dir, class_map=None):
    """
    Convert COCO annotations to segmentation mask PNG (clean version)
    """

   
    reset_folder(output_dir)

    
    coco = COCO(coco_json_path)

    print("\nCategories:")
    for cat in coco.loadCats(coco.getCatIds()):
        print(cat)

   
    for img_id in coco.getImgIds():

        img_info = coco.loadImgs(img_id)[0]
        h, w = img_info["height"], img_info["width"]

        # background = 0
        mask = np.zeros((h, w), dtype=np.uint8)

        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)

        for ann in anns:

            category_id = ann["category_id"]
            ann_mask = coco.annToMask(ann)

            
            if class_map is not None:
                if category_id not in class_map:
                    continue
                label = class_map[category_id]
            else:
                label = category_id

            mask[ann_mask == 1] = label

    
        save_name = os.path.splitext(img_info["file_name"])[0] + ".png"

        cv2.imwrite(
            os.path.join(output_dir, save_name),
            mask
        )

    print(f"\nDone → Mask saved to {output_dir}")

In [ ]:
class_map = {
    1: 1,  # Nekrosis
    2: 2   # Pulpitis
}

In [ ]:
import json

def check_json(path):
    with open(path, "r") as f:
        data = json.load(f)

    print(path)
    print("Images     :", len(data["images"]))
    print("Annotations:", len(data["annotations"]))
    print("-"*30)

check_json("/kaggle/input/datasets/zahra313/dataskripsizahracscoco/train/_annotations.coco.json")
check_json("/kaggle/input/datasets/zahra313/dataskripsizahracscoco/valid/_annotations.coco.json")
check_json("/kaggle/input/datasets/zahra313/dataskripsizahracscoco/test/_annotations.coco.json")

### Membuat Mask Train

In [ ]:
train_json = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco/train/_annotations.coco.json"

coco_to_mask(
    coco_json_path=train_json,
    output_dir="/kaggle/working/train_masks",
    class_map=class_map
)

### Membuat Mask Valid

In [ ]:
valid_json = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco/valid/_annotations.coco.json"

coco_to_mask(
    coco_json_path=valid_json,
    output_dir="/kaggle/working/valid_masks",
    class_map=class_map
)

### Membuat Mask Test

In [ ]:
test_json = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco/test/_annotations.coco.json"

coco_to_mask(
    coco_json_path=test_json,
    output_dir="/kaggle/working/test_masks",
    class_map=class_map
)

### Verifikasi Mask

In [ ]:
import os

folders = {
    "Train": "/kaggle/working/train_masks",
    "Valid": "/kaggle/working/valid_masks",
    "Test": "/kaggle/working/test_masks"
}

for name, path in folders.items():
    print(f"{name} Mask : {len(os.listdir(path))}")

### Cek Jumlah Mask per Kelas

In [ ]:
import os
import cv2
import numpy as np

mask_dir = "/kaggle/working/train_masks"

nekrosis_count = 0
pulpitis_count = 0

for mask_file in os.listdir(mask_dir):

    mask = cv2.imread(
        os.path.join(mask_dir, mask_file),
        0
    )

    unique_values = np.unique(mask)

    if 1 in unique_values:
        nekrosis_count += 1

    if 2 in unique_values:
        pulpitis_count += 1

print(f"Nekrosis : {nekrosis_count} mask")
print(f"Pulpitis : {pulpitis_count} mask")

## Membuat Struktur Dataset U-Net

### Dataset Class

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset

class DentalDataset(Dataset):

    def __init__(self, image_dir, mask_dir):

        self.image_dir = image_dir
        self.mask_dir = mask_dir

        self.images = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image_name = self.images[idx]

        image_path = os.path.join(self.image_dir, image_name)
        mask_name = os.path.splitext(image_name)[0] + ".png"
        mask_path = os.path.join(self.mask_dir, mask_name)

      
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Image not found: {image_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = image.astype(np.float32) / 255.0

        image = torch.from_numpy(image).permute(2, 0, 1)

       
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise ValueError(f"Mask not found: {mask_path}")

        mask = torch.from_numpy(mask).long()

        return image, mask

### Membuat Dataset

In [ ]:
train_dataset = DentalDataset(
    image_dir="/kaggle/input/datasets/zahra313/dataskripsizahracscoco/train",
    mask_dir="/kaggle/working/train_masks")

In [ ]:
import os

img_dir = "/kaggle/input/datasets/zahra313/dataskripsizahracscoco/train"
mask_dir = "/kaggle/working/train_masks"

images = sorted([
    os.path.splitext(f)[0]
    for f in os.listdir(img_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

masks = sorted([
    os.path.splitext(f)[0]
    for f in os.listdir(mask_dir)
])

print("Images:", len(images))
print("Masks :", len(masks))

print("Sample image:", images[:3])
print("Sample mask :", masks[:3])

In [ ]:
print("Jumlah data:", len(train_dataset))

image, mask = train_dataset[0]

print("Image:", image.shape)
print("Mask :", mask.shape)
print("Label:", torch.unique(mask))

In [ ]:
val_dataset = DentalDataset(
    image_dir="/kaggle/input/datasets/zahra313/dataskripsizahracscoco/valid",
    mask_dir="/kaggle/working/valid_masks")

In [ ]:
print("Jumlah data:", len(val_dataset))

image, mask = val_dataset[0]

print("Image:", image.shape)
print("Mask :", mask.shape)
print("Label:", torch.unique(mask))

In [ ]:
test_dataset = DentalDataset(
    image_dir="/kaggle/input/datasets/zahra313/dataskripsizahracscoco/test",
    mask_dir="/kaggle/working/test_masks")

In [ ]:
print("Jumlah data:", len(test_dataset))

image, mask = test_dataset[0]

print("Image:", image.shape)
print("Mask :", mask.shape)
print("Label:", torch.unique(mask))

### Membuat DataLoader

In [ ]:
import torch
import numpy as np
import random

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
images, masks = next(iter(train_loader))

print(images.shape)
print(masks.shape)

In [ ]:
images, masks = next(iter(valid_loader))

print(images.shape)
print(masks.shape)

In [ ]:
images, masks = next(iter(test_loader))

print(images.shape)
print(masks.shape)

## Modeling

In [ ]:
pip install -q segmentation-models-pytorch

In [ ]:
import torch
import segmentation_models_pytorch as smp

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")

print(device)

### Membuat Model U-Net

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=3
)

model = model.to(device)

### Uji Forward Pass

In [ ]:
images, masks = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():
    outputs = model(images)

print("Input :", images.shape)
print("Output:", outputs.shape)

### Loss Function

In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn

dice_loss = smp.losses.DiceLoss(
    mode="multiclass"
)

ce_loss = nn.CrossEntropyLoss()

def loss_fn(pred, target):
    return ce_loss(pred, target) + dice_loss(pred, target)

### Optimizer

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001)

### Metric Dice Score

In [ ]:
import torch

def dice_score(pred, target, smooth=1e-6):

    pred = torch.argmax(pred, dim=1)

    pred = pred.reshape(-1)
    target = target.reshape(-1)

    intersection = (pred == target).float().sum()

    return (
        2 * intersection + smooth
    ) / (
        pred.numel() + target.numel() + smooth
    )

### Metric IoU

In [ ]:
def iou_score(pred, target, smooth=1e-6):

    pred = torch.argmax(pred, dim=1)

    pred = pred.reshape(-1)
    target = target.reshape(-1)

    intersection = (pred == target).float().sum()

    union = pred.numel() + target.numel() - intersection

    return (
        intersection + smooth
    ) / (
        union + smooth
    )

### Cek Loss Sekali

In [ ]:
images, masks = next(iter(train_loader))

images = images.to(device)
masks = masks.to(device)

outputs = model(images)

loss = loss_fn(outputs, masks)

print("Loss :", loss.item())

## Training

### Variabel Training

In [ ]:
import torch
import numpy as np

num_epochs = 100
best_val_loss = float("inf")
patience = 10
counter = 0
train_losses = []
val_losses = []
train_dices = []
val_dices = []
train_ious = []
val_ious = []

### Fungsi Metric

In [ ]:
def dice_score(pred, target, smooth=1e-6):

    pred = torch.argmax(pred, dim=1)
    pred = pred.reshape(-1)
    target = target.reshape(-1)
    intersection = (pred == target).float().sum()
    return (
        2 * intersection + smooth
    ) / (
        pred.numel() + target.numel() + smooth
    )

def iou_score(pred, target, smooth=1e-6):

    pred = torch.argmax(pred, dim=1)
    pred = pred.reshape(-1)
    target = target.reshape(-1)
    intersection = (pred == target).float().sum()
    union = pred.numel() + target.numel() - intersection
    return (
        intersection + smooth
    ) / (
        union + smooth
    )

### Training + Validation + Best Model + Early Stopping

In [ ]:
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter


writer = SummaryWriter("runs/unet_experiment")

for epoch in range(num_epochs):

   
    model.train()

    train_loss = 0.0
    train_dice = 0.0
    train_iou = 0.0

    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")

    for images, masks in train_loop:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_fn(outputs, masks)

        loss.backward()
        optimizer.step()

       
        batch_loss = loss.item()
        batch_dice = dice_score(outputs, masks).item()
        batch_iou = iou_score(outputs, masks).item()

        train_loss += batch_loss
        train_dice += batch_dice
        train_iou += batch_iou

        # update progress bar
        train_loop.set_postfix(loss=batch_loss, dice=batch_dice, iou=batch_iou)

    avg_train_loss = train_loss / len(train_loader)
    avg_train_dice = train_dice / len(train_loader)
    avg_train_iou = train_iou / len(train_loader)


    model.eval()

    val_loss = 0.0
    val_dice = 0.0
    val_iou = 0.0

    val_loop = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")

    with torch.no_grad():
        for images, masks in val_loop:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = loss_fn(outputs, masks)

            batch_loss = loss.item()
            batch_dice = dice_score(outputs, masks).item()
            batch_iou = iou_score(outputs, masks).item()

            val_loss += batch_loss
            val_dice += batch_dice
            val_iou += batch_iou

            val_loop.set_postfix(loss=batch_loss, dice=batch_dice, iou=batch_iou)

    avg_val_loss = val_loss / len(valid_loader)
    avg_val_dice = val_dice / len(valid_loader)
    avg_val_iou = val_iou / len(valid_loader)

    
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    train_dices.append(avg_train_dice)
    val_dices.append(avg_val_dice)

    train_ious.append(avg_train_iou)
    val_ious.append(avg_val_iou)

    writer.add_scalar("Loss/Train", avg_train_loss, epoch)
    writer.add_scalar("Loss/Val", avg_val_loss, epoch)

    writer.add_scalar("Dice/Train", avg_train_dice, epoch)
    writer.add_scalar("Dice/Val", avg_val_dice, epoch)

    writer.add_scalar("IoU/Train", avg_train_iou, epoch)
    writer.add_scalar("IoU/Val", avg_val_iou, epoch)

   
    print(
        f"\nEpoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss={avg_train_loss:.4f} | Val Loss={avg_val_loss:.4f} | "
        f"Train Dice={avg_train_dice:.4f} | Val Dice={avg_val_dice:.4f} | "
        f"Train IoU={avg_train_iou:.4f} | Val IoU={avg_val_iou:.4f}"
    )

    
    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        torch.save(
            model.state_dict(),
            "/kaggle/working/best_unet.pth"
        )

        counter = 0
        print("✓ Best model saved")

    else:
        counter += 1


    if counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

writer.close()

### Plot Hasil Training

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(18,5))

# Loss
plt.subplot(1,3,1)
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Validation")
plt.title("Loss")
plt.legend()

# Dice
plt.subplot(1,3,2)
plt.plot(train_dices, label="Train")
plt.plot(val_dices, label="Validation")
plt.title("Dice Score")
plt.legend()

# IoU
plt.subplot(1,3,3)
plt.plot(train_ious, label="Train")
plt.plot(val_ious, label="Validation")
plt.title("IoU")
plt.legend()

plt.tight_layout()
plt.show()

## Load Best Model Sebelum Testing

In [ ]:
model.load_state_dict(
    torch.load(
        "/kaggle/working/best_unet.pth"
    )
)

model.eval()

In [ ]:
import torch
import numpy as np
from sklearn.metrics import confusion_matrix
from tqdm import tqdm

def evaluate(model, loader, loss_fn, device, num_classes=3):
    model.eval()
    total_loss = 0
    y_true = []
    y_pred = []

    with torch.no_grad():
        loop = tqdm(loader, desc="Testing")
        for images, masks in loop:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)

            y_true.append(
                masks.cpu().numpy().flatten())

            y_pred.append(
                preds.cpu().numpy().flatten())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(num_classes)))

    iou_per_class = []
    dice_per_class = []
    precision_per_class = []
    recall_per_class = []

    for cls in range(num_classes):
        TP = cm[cls, cls]
        FP = cm[:, cls].sum() - TP
        FN = cm[cls, :].sum() - TP
        TN = cm.sum() - TP - FP - FN
        iou = TP / (TP + FP + FN + 1e-6)
        dice = (2 * TP) / (
            2 * TP + FP + FN + 1e-6)
        precision = TP / (TP + FP + 1e-6)
        recall = TP / (TP + FN + 1e-6)
        iou_per_class.append(iou)
        dice_per_class.append(dice)
        precision_per_class.append(precision)
        recall_per_class.append(recall)

    iou_per_class = np.array(iou_per_class)
    dice_per_class = np.array(dice_per_class)
    precision_per_class = np.array(precision_per_class)
    recall_per_class = np.array(recall_per_class)
    mean_iou = iou_per_class.mean()
    mean_dice = dice_per_class.mean()
    mean_precision = precision_per_class.mean()
    mean_recall = recall_per_class.mean()

    class_names = [
        "Background",
        "Nekrosis",
        "Pulpitis"]
    print("\n===== FINAL TEST RESULT =====")
    print(f"Loss           : {total_loss/len(loader):.4f}")
    print(f"Mean IoU       : {mean_iou:.4f}")
    print(f"Mean Dice      : {mean_dice:.4f}")
    print(f"Mean Precision : {mean_precision:.4f}")
    print(f"Mean Recall    : {mean_recall:.4f}")
    print("\n===== PER CLASS =====")
    for i in range(num_classes):
        print(
            f"{class_names[i]:12s} "
            f"IoU={iou_per_class[i]:.4f} "
            f"Dice={dice_per_class[i]:.4f} "
            f"Prec={precision_per_class[i]:.4f} "
            f"Recall={recall_per_class[i]:.4f}")
    return {
        "confusion_matrix": cm,
        "iou_per_class": iou_per_class,
        "dice_per_class": dice_per_class,
        "precision_per_class": precision_per_class,
        "recall_per_class": recall_per_class,
        "mean_iou": mean_iou,
        "mean_dice": mean_dice,
        "mean_precision": mean_precision,
        "mean_recall": mean_recall}

## Evaluasi Model

### Evaluasi Model menggunakan data Validasi

In [ ]:
results = evaluate(
    model=model,
    loader=valid_loader,
    loss_fn=loss_fn,
    device=device,
    num_classes=3
)

### Evaluasi Model menggunakan data Test

In [ ]:
results = evaluate(
    model=model,
    loader=test_loader,
    loss_fn=loss_fn,
    device=device,
    num_classes=3
)

## Membuat mask untuk seluruh data test

In [ ]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm

output_dir = "/kaggle/working/predicted_masks"
os.makedirs(output_dir, exist_ok=True)

model.eval()

with torch.no_grad():

    for idx in tqdm(range(len(test_dataset))):

        image, _ = test_dataset[idx]

        image_input = image.unsqueeze(0).to(device)

        output = model(image_input)

        pred_mask = torch.argmax(
            output,
            dim=1
        ).squeeze().cpu().numpy()

        mask_name = test_dataset.images[idx]
        mask_name = os.path.splitext(mask_name)[0] + ".png"

        cv2.imwrite(
            os.path.join(output_dir, mask_name),
            pred_mask.astype(np.uint8)
        )

print("Semua mask berhasil disimpan.")

### Mengecek jumlah mask hasil prediksi

In [ ]:
print(
    "Jumlah mask:",
    len(os.listdir("/kaggle/working/predicted_masks")))

### Menampilkan hasil mask

In [ ]:
import matplotlib.pyplot as plt

image, gt_mask = test_dataset[1]

model.eval()

with torch.no_grad():

    pred = model(
        image.unsqueeze(0).to(device)
    )

pred_mask = torch.argmax(
    pred,
    dim=1
).squeeze().cpu().numpy()

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(image.permute(1,2,0))
plt.title("Original Image")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(gt_mask)
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(pred_mask)
plt.title("Predicted Mask")
plt.axis("off")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

model.eval()

n = len(test_dataset)

fig, axes = plt.subplots(
    n,
    4,
    figsize=(16, n*3))

with torch.no_grad():
    for i in range(n):
        image, gt_mask = test_dataset[i]

        pred = model(
            image.unsqueeze(0).to(device))

        pred_mask = torch.argmax(
            pred,
            dim=1
        ).squeeze().cpu().numpy()

        # Tensor -> NumPy
        image_np = image.permute(1,2,0).cpu().numpy()
        # ==========================
        # OVERLAY MASK KE GAMBAR
        # ==========================

        overlay = image_np.copy()

        # Nekrosis = merah
        overlay[pred_mask == 1] = [
            1.0, 0.0, 0.0]

        # Pulpitis = hijau
        overlay[pred_mask == 2] = [
            0.0, 1.0, 0.0]

        # ==========================
        # ORIGINAL IMAGE
        # ==========================
        axes[i,0].imshow(image_np)
        axes[i,0].set_title("Original")

        # ==========================
        # GROUND TRUTH
        # ==========================
        axes[i,1].imshow(gt_mask)
        axes[i,1].set_title("Ground Truth")

        # ==========================
        # PREDICTED MASK
        # ==========================
        axes[i,2].imshow(pred_mask)
        axes[i,2].set_title("Prediction")

        # ==========================
        # OVERLAY
        # ==========================
        axes[i,3].imshow(overlay)
        axes[i,3].set_title("Overlay")

        for j in range(4):
            axes[i,j].axis("off")

plt.tight_layout()
plt.show()

Keterangan:
| Label | Kelas      | Warna    |
| ----- | ---------- | -------- |
| 0     | Background | ⚫ Hitam  |
| 1     | Nekrosis   | 🔴 Merah |
| 2     | Pulpitis   | 🟢 Hijau |


## Menyimpan Seluruh Hasil Ke lokal

In [ ]:
!ls -lh /kaggle/working

In [ ]:
!zip -r hasil_unet_lengkap.zip \
/kaggle/working/best_unet.pth \
/kaggle/working/predicted_masks \
/kaggle/working/runs \
/kaggle/working/train_masks \
/kaggle/working/valid_masks \
/kaggle/working/test_masks